In [1]:
import pandas as pd
from sklearn.model_selection import GroupKFold
import os

In [2]:
cohort = "luad"

data_path = '../../data/clinical_data'
filename = f'{cohort}_clinical'
n_folds = 5

In [3]:
# loading data

df = pd.read_csv(f"{data_path}/{filename}.csv", index_col=0)
df.head()

,case_id,dss_survival_days,dss_censorship
0,TCGA-05-4244,0.0,1
1,TCGA-05-4249,1523.0,1
3,TCGA-05-4382,607.0,1
4,TCGA-05-4384,426.0,1
5,TCGA-05-4389,1369.0,1


In [4]:
# identifying "location" from patient_id
df['location'] = df['case_id'].apply(lambda x: x.split('-')[1])

In [5]:
# tylko prawdziwe zdarzenia
events_df = df[df['dss_censorship'] == 0]

# maksymalny czas zdarzenia
max_event_time = events_df['dss_survival_days'].max()

# wszystkie obserwacje z tym czasem
critical_idx = events_df[
    events_df['dss_survival_days'] == max_event_time
].index


In [6]:
df_rest = df.drop(index=critical_idx).reset_index(drop=True)
critical_df = df.loc[critical_idx].reset_index(drop=True)


In [7]:


gkf = GroupKFold(n_splits=n_folds)

# creating a folder for splits
split_dir = f"{data_path}/{filename}/splits"
os.makedirs(split_dir, exist_ok=True)

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(df_rest, groups=df_rest['location'])
):
    fold_path = os.path.join(split_dir, f"{fold}")
    os.makedirs(fold_path, exist_ok=True)

    train_df = df_rest.iloc[train_idx].reset_index(drop=True)
    test_df = df_rest.iloc[test_idx].reset_index(drop=True)

    # 🔥 KLUCZOWY KROK: zawsze dokładamy krytyczne obserwacje do train
    # train_df = pd.concat([train_df, critical_df], ignore_index=True)

    # tylko eventy
    train_events = train_df[train_df['dss_censorship'] == 0]
    test_events = test_df[test_df['dss_censorship'] == 0]

    if len(test_events) > 0:
        max_train_event_time = train_events['dss_survival_days'].max()
        problematic = test_events[
            test_events['dss_survival_days'] > max_train_event_time
        ]

        if len(problematic) > 0:
            # 🔥 przenosimy je z test → train
            train_df = pd.concat([train_df, problematic], ignore_index=True)
            test_df = test_df.drop(problematic.index).reset_index(drop=True)


    train_df.to_csv(os.path.join(fold_path, "train_filtered.csv"), index=False)
    test_df.to_csv(os.path.join(fold_path, "test_filtered.csv"), index=False)

    print(
        f"Fold {fold}: "
        f"train={len(train_df)}, "
        f"test={len(test_df)}, "
        f"critical_in_train={len(critical_df)}"
    )


Fold 0: train=379, test=95, critical_in_train=1
Fold 1: train=379, test=95, critical_in_train=1
Fold 2: train=379, test=95, critical_in_train=1
Fold 3: train=379, test=95, critical_in_train=1
Fold 4: train=381, test=93, critical_in_train=1


In [8]:
# # identifying "location" from patient_id
# df['location'] = df['case_id'].apply(lambda x: x.split('-')[1])

# # creating a folder for splits
# split_dir = f"{data_path}/{filename}/splits"
# os.makedirs(split_dir, exist_ok=True)

# # GroupKFold by location
# gkf = GroupKFold(n_splits=n_folds)

# for fold, (train_idx, test_idx) in enumerate(gkf.split(df, groups=df['location'])):
#     fold_path = os.path.join(split_dir, f"{fold}")
#     os.makedirs(fold_path, exist_ok=True)

#     train_df = df.iloc[train_idx].reset_index(drop=True)
#     test_df = df.iloc[test_idx].reset_index(drop=True)

#     train_df.to_csv(os.path.join(fold_path, "train_filtered.csv"), index=False)
#     test_df.to_csv(os.path.join(fold_path, "test_filtered.csv"), index=False)

#     print(f"Fold {fold}: train={len(train_df)}, test={len(test_df)}")